# SmartRoad AI — Entrenamiento YOLO11n en Google Colab (GPU)

Este notebook entrena **YOLO11n** sobre el dataset SmartRoad AI (6 clases: `Accidente`, `Bus`, `Camion`, `Carro`, `Fire`, `Moto`) usando la GPU de Colab, y deja todo en Google Drive con la **misma estructura del repositorio** (`experiments/yolov11/...` y `evaluation/yolov11/...`), lista para copiarla al repo y subirla a GitHub.

**Antes de empezar:**
1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.
2. Tener `dataset.zip` subido en `Mi unidad/SmartRoadAI/dataset.zip`.

Si Colab se desconecta, **vuelve a ejecutar todas las celdas en orden**: la celda de entrenamiento detecta el `last.pt` en Drive y **reanuda** desde la última época.

In [ ]:
# 1. Verificar que hay GPU
!nvidia-smi

In [ ]:
# 2. Montar Google Drive (aquí se guardan los resultados para que no se pierdan si Colab se cae)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Configuración (solo cambia esto si es necesario)
from pathlib import Path

DRIVE_BASE      = Path('/content/drive/MyDrive/SmartRoadAI')   # carpeta en tu Drive
DATASET_ZIP     = DRIVE_BASE / 'dataset.zip'                   # zip de la carpeta dataset/ del repo

FAMILIA         = 'yolov11'                                    # carpeta de la familia (igual que en el repo)
EXPERIMENT_NAME = 'entrenamiento_02_20_epochs_colab'           # nombre ÚNICO (el 01 ya existe en el repo)
EVAL_NAME       = 'test_final'

EXPERIMENTS_DIR = DRIVE_BASE / 'experiments' / FAMILIA          # = project de model.train()
EVALUATION_DIR  = DRIVE_BASE / 'evaluation'  / FAMILIA          # = project de model.val()
EXPERIMENT_DIR  = EXPERIMENTS_DIR / EXPERIMENT_NAME

LOCAL_DATASET   = Path('/content/dataset')                      # disco local de Colab (rápido)
BASE_WEIGHTS    = Path('/content/models/base/yolo11n.pt')

# Hiperparámetros (mismos epochs/imgsz/seed que YOLOv8 y YOLOv10 para que la comparación sea justa)
EPOCHS   = 20
IMG_SIZE = 640
BATCH    = 16      # en T4 cabe bien; si da "CUDA out of memory" bájalo a 8
WORKERS  = 2

assert DATASET_ZIP.exists(), f'No encuentro {DATASET_ZIP}. Sube dataset.zip a esa ruta en Drive.'
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
print('OK. Resultados en:', EXPERIMENT_DIR)

In [ ]:
# 4. Instalar Ultralytics
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# 5. Copiar y descomprimir el dataset en el disco local de Colab
#    (entrenar leyendo directo desde Drive es MUY lento)
import shutil, zipfile, os

if not (LOCAL_DATASET / 'train' / 'images').exists():
    print('Copiando zip desde Drive...')
    shutil.copy(DATASET_ZIP, '/content/dataset.zip')
    print('Descomprimiendo...')
    with zipfile.ZipFile('/content/dataset.zip') as z:
        z.extractall('/content/_unzip')
    # El zip puede traer la carpeta 'dataset/' adentro o directamente train/valid/test
    yamls = list(Path('/content/_unzip').rglob('data.yaml'))
    assert yamls, 'El zip no contiene data.yaml'
    shutil.move(str(yamls[0].parent), str(LOCAL_DATASET))
    shutil.rmtree('/content/_unzip', ignore_errors=True)
    os.remove('/content/dataset.zip')

# Borrar caches .cache generadas en Windows (Ultralytics las regenera)
for c in LOCAL_DATASET.rglob('*.cache'):
    c.unlink()

for split in ['train', 'valid', 'test']:
    n_img = len(list((LOCAL_DATASET / split / 'images').glob('*')))
    n_lbl = len(list((LOCAL_DATASET / split / 'labels').glob('*.txt')))
    print(f'{split:5s}: {n_img} imágenes / {n_lbl} labels')
# Esperado: train 37106 / valid 5305 / test 2648

In [ ]:
# 6. Crear un data.yaml con rutas absolutas de Colab (el original del repo NO se modifica)
import yaml

with open(LOCAL_DATASET / 'data.yaml', encoding='utf-8') as f:
    original = yaml.safe_load(f)

colab_yaml = {
    'path':  str(LOCAL_DATASET),
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    original['nc'],
    'names': original['names'],
}
DATA_YAML = Path('/content/data_colab.yaml')
with open(DATA_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump(colab_yaml, f, allow_unicode=True, sort_keys=False)

print(DATA_YAML.read_text())

In [ ]:
# 7. Peso base YOLO11n
import urllib.request
BASE_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
if not BASE_WEIGHTS.exists():
    urllib.request.urlretrieve(
        'https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt', BASE_WEIGHTS)
print(BASE_WEIGHTS, BASE_WEIGHTS.stat().st_size // 1024, 'KB')

In [ ]:
# 8. ENTRENAMIENTO (o reanudación automática si Colab se desconectó)
from ultralytics import YOLO

last_pt = EXPERIMENT_DIR / 'weights' / 'last.pt'

if last_pt.exists():
    print('Se encontró last.pt -> REANUDANDO entrenamiento desde', last_pt)
    model = YOLO(str(last_pt))
    model.train(resume=True)
else:
    assert not EXPERIMENT_DIR.exists() or not any(EXPERIMENT_DIR.iterdir()), \
        f'{EXPERIMENT_DIR} ya existe sin last.pt. Cambia EXPERIMENT_NAME para no sobrescribir evidencia.'
    model = YOLO(str(BASE_WEIGHTS))
    model.train(
        data=str(DATA_YAML),
        project=str(EXPERIMENTS_DIR),
        name=EXPERIMENT_NAME,
        exist_ok=True,          # la carpeta puede existir vacía si falló antes de la 1ª época
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=0,
        workers=WORKERS,
        amp=True,
        val=True,
        save=True,
        save_period=1,          # guarda epochN.pt cada época (igual que YOLOv10)
        patience=10,
        seed=0,
        plots=True,
        verbose=True,
    )

print('Entrenamiento terminado. Resultados en:', EXPERIMENT_DIR)

In [ ]:
# 9. Resumen de métricas del entrenamiento
import pandas as pd
df = pd.read_csv(EXPERIMENT_DIR / 'results.csv')
df.columns = df.columns.str.strip()
cols = [c for c in df.columns if c in ('epoch','metrics/precision(B)','metrics/recall(B)','metrics/mAP50(B)','metrics/mAP50-95(B)')]
display(df[cols].tail(10))
best = df.loc[df['metrics/mAP50-95(B)'].idxmax()]
print(f"Mejor época: {int(best['epoch'])}  mAP50={best['metrics/mAP50(B)']:.3f}  mAP50-95={best['metrics/mAP50-95(B)']:.3f}")

In [ ]:
# 10. Evaluación final sobre TEST -> evaluation/yolov11/test_final
best_pt = EXPERIMENT_DIR / 'weights' / 'best.pt'
assert best_pt.exists(), 'No existe best.pt'
assert not (EVALUATION_DIR / EVAL_NAME).exists(), 'Ya existe una evaluación con ese nombre; cambia EVAL_NAME.'

model = YOLO(str(best_pt))
metrics = model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    workers=WORKERS,
    plots=True,
    verbose=True,
    project=str(EVALUATION_DIR),
    name=EVAL_NAME,
    exist_ok=False,
)

# Guardar un resumen en texto con las métricas por clase
lines = [
    'SMARTROAD AI - YOLO11n - Evaluacion TEST',
    f'Modelo: {EXPERIMENT_NAME}/weights/best.pt',
    f'Precision: {metrics.box.mp:.4f}',
    f'Recall:    {metrics.box.mr:.4f}',
    f'mAP50:     {metrics.box.map50:.4f}',
    f'mAP50-95:  {metrics.box.map:.4f}',
    '', 'Por clase (mAP50-95):',
]
for i, c in enumerate(metrics.box.ap_class_index):
    lines.append(f'  {model.names[int(c)]:10s} {metrics.box.maps[int(c)]:.4f}')
(EVALUATION_DIR / EVAL_NAME / 'metrics_summary.txt').write_text('\n'.join(lines), encoding='utf-8')
print('\n'.join(lines))

In [ ]:
# 11. Predicciones visuales (misma selección que los scripts de YOLOv10: seed 42, 30 generales y 20 de Fire)
import random

test_imgs = sorted((LOCAL_DATASET / 'test' / 'images').glob('*'))
pred_project = EVALUATION_DIR / EVAL_NAME / 'predictions'

# 11a. Muestras generales
muestras = random.Random(42).sample(test_imgs, 30)
model.predict(source=[str(p) for p in muestras], imgsz=IMG_SIZE, conf=0.25, iou=0.70, device=0,
              save=True, show_conf=True, project=str(pred_project), name='muestras_generales', exist_ok=False)

# 11b. Prueba dirigida de Fire (class_id = 4)
def tiene_fire(img):
    lbl = LOCAL_DATASET / 'test' / 'labels' / (img.stem + '.txt')
    if not lbl.exists():
        return False
    return any(l.split() and int(float(l.split()[0])) == 4 for l in lbl.read_text().splitlines())

fire_imgs = [p for p in test_imgs if tiene_fire(p)]
print('Imágenes TEST con Fire:', len(fire_imgs))
fire_sel = random.Random(42).sample(fire_imgs, min(20, len(fire_imgs)))
model.predict(source=[str(p) for p in fire_sel], imgsz=IMG_SIZE, conf=0.25, iou=0.70, device=0,
              save=True, show_conf=True, project=str(pred_project / 'fire_test'), name='results', exist_ok=False)

In [ ]:
# 12. Empaquetar resultados en un ZIP (para descargarlos y meterlos al repositorio)
# Comprimimos solo experiments/yolov11/<experimento> y evaluation/yolov11/test_final,
# respetando las rutas relativas del repo
import zipfile
zip_path = DRIVE_BASE / 'resultados_yolov11_colab.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in [EXPERIMENT_DIR, EVALUATION_DIR / EVAL_NAME]:
        for f in folder.rglob('*'):
            if f.is_file():
                z.write(f, f.relative_to(DRIVE_BASE))
print('ZIP listo en Drive:', zip_path, f'({zip_path.stat().st_size/1e6:.1f} MB)')
print('Descárgalo desde drive.google.com -> Mi unidad -> SmartRoadAI')

## Siguiente paso
Descarga `resultados_yolov11_colab.zip` desde Drive y descomprímelo **dentro de la raíz del repositorio** `Entrenamiento_DataScience/`. Quedarán:

- `experiments/yolov11/entrenamiento_02_20_epochs_colab/` (pesos, results.csv, gráficas)
- `evaluation/yolov11/test_final/` (métricas TEST, matrices de confusión, predicciones)

Luego `git add` + `commit` + `push` en la rama `yolo11-jorel`.